In [22]:
import pandas as pd
import sqlite3
import os

df = pd.read_csv('../outputs/companies_clean.csv')
print("✓ Loaded:", df.shape)
print(df.head())

✓ Loaded: (15, 7)
  Company  Year  Sales  Expenses  Operating Profit  Net profit  OPM
0  Zomato  FY21   1994      3403             -1409        -816  -71
1  Zomato  FY22   4192      6499             -2307       -1222  -55
2  Zomato  FY23   7079      8693             -1614        -971  -23
3  Zomato  FY24  12114     12294              -180         351   -1
4  Zomato  FY25  17003     15337              1666         527   10


In [23]:
conn = sqlite3.connect('../outputs/business_health.db')
df.to_sql('financials', conn, if_exists='replace', index=False)
print("✓ Loaded into SQLite")

result = pd.read_sql("SELECT Company, Year, Sales, [Net profit], OPM FROM financials LIMIT 9", conn)
print(result)

✓ Loaded into SQLite
  Company  Year  Sales  Net profit  OPM
0  Zomato  FY21   1994        -816  -71
1  Zomato  FY22   4192       -1222  -55
2  Zomato  FY23   7079        -971  -23
3  Zomato  FY24  12114         351   -1
4  Zomato  FY25  17003         527   10
5   Nykaa  FY21   2453          62    3
6   Nykaa  FY22   3774         -20    1
7   Nykaa  FY23   5144           9    4
8   Nykaa  FY24   6386          40    4


In [24]:
query_growth = """
SELECT
    Company, Year, Sales,
    LAG(Sales) OVER (PARTITION BY Company ORDER BY Year) AS prev_sales,
    ROUND(
        100.0 * (Sales - LAG(Sales) OVER (PARTITION BY Company ORDER BY Year))
        / LAG(Sales) OVER (PARTITION BY Company ORDER BY Year), 1
    ) AS yoy_growth_pct
FROM financials
WHERE Sales IS NOT NULL
ORDER BY Company, Year
"""
df_growth = pd.read_sql(query_growth, conn)
print(df_growth)

   Company  Year  Sales  prev_sales  yoy_growth_pct
0    Nykaa  FY21   2453         NaN             NaN
1    Nykaa  FY22   3774      2453.0            53.9
2    Nykaa  FY23   5144      3774.0            36.3
3    Nykaa  FY24   6386      5144.0            24.1
4    Nykaa  FY25   7754      6386.0            21.4
5    Paytm  FY21   2802         NaN             NaN
6    Paytm  FY22   4974      2802.0            77.5
7    Paytm  FY23   7990      4974.0            60.6
8    Paytm  FY24   9978      7990.0            24.9
9    Paytm  FY25  10483      9978.0             5.1
10  Zomato  FY21   1994         NaN             NaN
11  Zomato  FY22   4192      1994.0           110.2
12  Zomato  FY23   7079      4192.0            68.9
13  Zomato  FY24  12114      7079.0            71.1
14  Zomato  FY25  17003     12114.0            40.4


In [25]:
query_cost = """
SELECT
    Company, Year, Sales, Expenses,
    ROUND(100.0 * Expenses / Sales, 1) AS cost_ratio_pct
FROM financials
WHERE Sales IS NOT NULL AND Expenses IS NOT NULL AND Sales > 0
ORDER BY Company, Year
"""
df_cost = pd.read_sql(query_cost, conn)
print(df_cost)

   Company  Year  Sales  Expenses  cost_ratio_pct
0    Nykaa  FY21   2453      2369            96.6
1    Nykaa  FY22   3774      3726            98.7
2    Nykaa  FY23   5144      4940            96.0
3    Nykaa  FY24   6386      6116            95.8
4    Nykaa  FY25   7754      7248            93.5
5    Paytm  FY21   2802      4826           172.2
6    Paytm  FY22   4974      7286           146.5
7    Paytm  FY23   7990      9609           120.3
8    Paytm  FY24   9978     11115           111.4
9    Paytm  FY25  10483     10298            98.2
10  Zomato  FY21   1994      3403           170.7
11  Zomato  FY22   4192      6499           155.0
12  Zomato  FY23   7079      8693           122.8
13  Zomato  FY24  12114     12294           101.5
14  Zomato  FY25  17003     15337            90.2


In [26]:
query_risk = """
SELECT
    Company, Year, OPM,
    LAG(OPM) OVER (PARTITION BY Company ORDER BY Year) AS prev_opm,
    CASE
        WHEN OPM < LAG(OPM) OVER (PARTITION BY Company ORDER BY Year)
        THEN 'RISK FLAG'
        ELSE 'OK'
    END AS margin_status
FROM financials
WHERE OPM IS NOT NULL
ORDER BY Company, Year
"""
df_risk = pd.read_sql(query_risk, conn)
print(df_risk)

   Company  Year  OPM  prev_opm margin_status
0    Nykaa  FY21    3       NaN            OK
1    Nykaa  FY22    1       3.0     RISK FLAG
2    Nykaa  FY23    4       1.0            OK
3    Nykaa  FY24    4       4.0            OK
4    Nykaa  FY25    7       4.0            OK
5    Paytm  FY21  -72       NaN            OK
6    Paytm  FY22  -46     -72.0            OK
7    Paytm  FY23  -20     -46.0            OK
8    Paytm  FY24  -11     -20.0            OK
9    Paytm  FY25    2     -11.0            OK
10  Zomato  FY21  -71       NaN            OK
11  Zomato  FY22  -55     -71.0            OK
12  Zomato  FY23  -23     -55.0            OK
13  Zomato  FY24   -1     -23.0            OK
14  Zomato  FY25   10      -1.0            OK


In [27]:
query_rank = """
SELECT
    Company, Year, Sales, [Net profit], OPM,
    RANK() OVER (ORDER BY OPM DESC) AS opm_rank,
    RANK() OVER (ORDER BY Sales DESC) AS revenue_rank
FROM financials
WHERE Year = 'FY25'
ORDER BY opm_rank
"""
df_rank = pd.read_sql(query_rank, conn)
print("\nLatest year rankings:")
print(df_rank)


Latest year rankings:
  Company  Year  Sales  Net profit  OPM  opm_rank  revenue_rank
0  Zomato  FY25  17003         527   10         1             1
1   Nykaa  FY25   7754          26    7         2             3
2   Paytm  FY25  10483        1422    2         3             2


In [28]:
df_growth.to_csv('../outputs/revenue_growth.csv', index=False)
df_risk.to_csv('../outputs/risk_flags.csv', index=False)
df_rank.to_csv('../outputs/company_rankings.csv', index=False)
print("✓ revenue_growth.csv saved")
print("✓ risk_flags.csv saved")
print("✓ company_rankings.csv saved")
conn.close()

✓ revenue_growth.csv saved
✓ risk_flags.csv saved
✓ company_rankings.csv saved
